# Customer Churn Prediction Model Training

This notebook trains an XGBoost model to predict customer churn with optimized hyperparameters and threshold tuning.

## 1. Import Required Libraries

In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Load and Prepare Data

In [3]:
# Load dataset
data_path = os.path.join("..", "data", "Telco_Customer_Churn_Model_Ready.csv")
df = pd.read_csv(data_path)

# Handle potential semicolon delimiter
if df.shape[1] == 1:
    df = pd.read_csv(data_path, sep=";")

# Standardize column names
df.columns = df.columns.str.strip().str.lower()

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

Dataset loaded: 7043 rows, 20 columns

Columns: ['gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']


## 3. Prepare Target Variable

In [4]:
# Identify target column
target_col = next(c for c in df.columns if "churn" in c)
print(f"Target column: {target_col}")

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Convert target to binary (0/1)
y = (
    y.astype(str)
     .str.strip()
     .str.lower()
     .isin(["yes", "1", "true"])
     .astype(int)
)

print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nChurn rate: {y.mean():.2%}")

Target column: churn

Target distribution:
churn
0    5174
1    1869
Name: count, dtype: int64

Churn rate: 26.54%


## 4. Split Data into Train, Validation, and Test Sets

- **Training set (70%)**: Used to train the model
- **Validation set (15%)**: Used for early stopping during training
- **Test set (15%)**: Used for final evaluation

In [5]:
# First split: 70% train, 30% temp (for val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

Training set: 5634 samples
Validation set: 704 samples
Test set: 705 samples


## 5. Create Preprocessing Pipeline

In [6]:
# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print(f"Numerical features ({len(num_cols)}): {num_cols}")
print(f"\nCategorical features ({len(cat_cols)}): {cat_cols}")

# Create preprocessor
preprocessor = ColumnTransformer([
    ("num", MinMaxScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

print("\n✓ Preprocessor created")

Numerical features (19): ['gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges']

Categorical features (0): []

✓ Preprocessor created


## 6. Calculate Class Imbalance Weight

In [7]:
# Random Oversampling applied after preprocessing on training data only
ros = RandomOverSampler(random_state=42)
print("✓ Random Oversampler configured")

✓ Random Oversampler configured


## 7. Configure XGBoost Model

In [8]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    
    # Boosting parameters
    n_estimators=1000,
    learning_rate=0.01,
    
    # Tree parameters
    max_depth=4,
    min_child_weight=3,
    
    # Sampling parameters
    subsample=0.8,
    colsample_bytree=0.8,
    
    # Regularization parameters
    gamma=0.3,
    reg_alpha=0.5,
    reg_lambda=1.0,
    
    early_stopping_rounds=50,
    
    # Other parameters
    random_state=42,
    n_jobs=-1
)

# Create full pipeline
pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", xgb)
])

print("✓ Model pipeline created")

✓ Model pipeline created


## 8. Train Model with Early Stopping

In [9]:
print("Training model...")

# Preprocess training dan validation data
X_train_prep = pipeline.named_steps['prep'].fit_transform(X_train)
X_val_prep = pipeline.named_steps['prep'].transform(X_val)

# Apply SMOTE hanya pada training data
# Apply Random Oversampling hanya pada training data
X_train_resampled, y_train_resampled = ros.fit_resample(X_train_prep, y_train)

print(f"Before Oversampling - Class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"After Oversampling  - Class distribution: {dict(zip(*np.unique(y_train_resampled, return_counts=True)))}")

# Train dengan data yang sudah di-resample
pipeline.named_steps['clf'].fit(
    X_train_resampled, y_train_resampled,
    eval_set=[(X_val_prep, y_val)],
    verbose=False
)

print("✓ Model training completed")
#print(f"Best iteration: {pipeline.named_steps['clf'].best_iteration}")

Training model...
Before Oversampling - Class distribution: {np.int64(0): np.int64(4139), np.int64(1): np.int64(1495)}
After Oversampling  - Class distribution: {np.int64(0): np.int64(4139), np.int64(1): np.int64(4139)}
✓ Model training completed


## 9. Threshold Tuning for Optimal F1-Score

In [12]:
# Transform test set and get predicted probabilities
X_test_prep = pipeline.named_steps['prep'].transform(X_test)
y_prob = pipeline.named_steps['clf'].predict_proba(X_test_prep)[:, 1]

# Find optimal threshold that maximizes F1-score
best_thr = 0.5
best_f1 = 0

thresholds = np.arange(0.3, 0.8, 0.01)
f1_scores = []

for t in thresholds:
    pred = (y_prob > t).astype(int)
    f1 = f1_score(y_test, pred)
    f1_scores.append(f1)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = t

print(f"Optimal threshold: {best_thr:.2f}")
print(f"F1-Score at optimal threshold: {best_f1:.4f}")

Optimal threshold: 0.45
F1-Score at optimal threshold: 0.6373


## 10. Final Predictions and Model Evaluation

In [13]:
# Make final predictions using optimal threshold
y_pred = (y_prob > best_thr).astype(int)

# Calculate all metrics
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=" * 50)
print("FINAL MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"\nPrecision: {precision:.4f} - {precision*100:.2f}% of predicted churns are correct")
print(f"Recall:    {recall:.4f} - {recall*100:.2f}% of actual churns are detected")
print(f"F1-Score:  {f1:.4f} - Balanced measure of precision and recall")
print(f"ROC-AUC:   {auc:.4f} - Overall model discrimination ability")

print("\n" + "=" * 50)
print("CONFUSION MATRIX")
print("=" * 50)
cm = confusion_matrix(y_test, y_pred)
print(f"\nTrue Negatives:  {cm[0,0]:4d} (Correctly predicted non-churn)")
print(f"False Positives: {cm[0,1]:4d} (Incorrectly predicted as churn)")
print(f"False Negatives: {cm[1,0]:4d} (Missed churns)")
print(f"True Positives:  {cm[1,1]:4d} (Correctly predicted churn)")

print("\n" + "=" * 50)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

FINAL MODEL PERFORMANCE METRICS

Precision: 0.5241 - 52.41% of predicted churns are correct
Recall:    0.8128 - 81.28% of actual churns are detected
F1-Score:  0.6373 - Balanced measure of precision and recall
ROC-AUC:   0.8382 - Overall model discrimination ability

CONFUSION MATRIX

True Negatives:   380 (Correctly predicted non-churn)
False Positives:  138 (Incorrectly predicted as churn)
False Negatives:   35 (Missed churns)
True Positives:   152 (Correctly predicted churn)

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

    No Churn       0.92      0.73      0.81       518
       Churn       0.52      0.81      0.64       187

    accuracy                           0.75       705
   macro avg       0.72      0.77      0.73       705
weighted avg       0.81      0.75      0.77       705



## 11. Save Predictions

In [ ]:
# Create DataFrame with predictions
predictions_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Probability': y_prob,
    'Correct': (y_test.values == y_pred).astype(int)
})

print("\nSample predictions:")
print(predictions_df.head(10))

# Save predictions to CSV
output_path = os.path.join("..", "models", "test_predictions.csv")
predictions_df.to_csv(output_path, index=False)
print(f"\n✓ Predictions saved to: {output_path}")